In [2]:
import os
import json

In [3]:
def triplet_prompt(data):
    prompts = []
    labels = []
    i = 1
    for key, values in data.items():
        entity, position, exist = values
        if entity is None and position is not None:
            prompt = f"{i}. Is the {position} normal?"
        elif entity is not None and position is None:
            prompt = f"{i}. Can you observe {entity} in this CT scan?"
        else:
            prompt = f"{i}. Is there {entity} in the {position}?"
        
        prompts.append(prompt)
        labels.append(str(i) + "." + str(exist))
        i+=1
    
    return "\n".join(prompts), "\n".join(labels)

In [ ]:
path = "/checkpoint/datasets.damaged/med-img-data/amosmm/imagesTrProcessed/amos_5005/text_triplet.json"
with open(path, 'r') as f:
    data = json.load(f)
r = triplet_prompt(data['abdomen'])

print(r[0])
print(r[1])

In [ ]:
path = "/scratch/ssd004/datasets/med-img-data/amosmm/dataset_withsplit.json"
with open(path) as f:
    data = json.load(f)

In [ ]:
to_save = "/scratch/ssd004/datasets/med-img-data/amosmm/imagesTrProcessed"
for i in data["training"]:
    image_name = i["image"].split("/")[-1].split(".")[0]
    if "splitted_findings" in i["labels"].keys(): 
        split_findings = i["labels"]["splitted_findings"]
    else:
        print("MISSING")
        continue
    save_to = to_save + "/" + image_name + "/split.json"

    with open(save_to, "w") as f:
        json.dump(
            split_findings,
            f
        )

In [ ]:
path = "/fs01/home/junma/MedicalVLM/Data/AMOSMM.json"
with open(path) as f:
    data = json.load(f)

for i, d in enumerate(data["train"]):
    # print(data['train'][i]["text"].replace("text", "split"))
    data['train'][i]["splitted_findings"] = data['train'][i]["text"].replace("text", "split")

with open(path, 'w') as f:
    json.dump(
        data,
        f
    )

In [4]:
import os
import json

def read_and_combine_json(directory):
    combined_data = []
    
    # Iterate through all files in the directory
    for root, _, files in os.walk(directory):
        for file in files:
            if file == "text_triplet.json":
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        data = json.load(f)
                        data["from"] = file_path
                        combined_data.append(data)
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")
    
    return combined_data

# Set the directory path
directory_path = "/checkpoint/datasets.damaged/med-img-data/amosmm/imagesTrProcessed"

# Read and combine JSON files
combined_list = read_and_combine_json(directory_path)


In [5]:
chest = {i['from']: list(i['pelvis'].values()) for i in combined_list if 'pelvis' in i.keys()}

In [25]:
chest

{'/checkpoint/datasets.damaged/med-img-data/amosmm/imagesTrProcessed/amos_7393/text_triplet.json': [['wall thickening',
   'bladder',
   False],
  ['enlarged', 'prostate', False],
  ['enlarged lymph nodes', 'pelvic cavity', False],
  ['fluid accumulation', 'pelvic cavity', False]],
 '/checkpoint/datasets.damaged/med-img-data/amosmm/imagesTrProcessed/amos_7392/text_triplet.json': [['thickening',
   'bladder wall',
   False],
  ['abnormal density', 'bladder cavity', False],
  ['enlargement', 'uterus', False],
  ['abnormal density', 'adnexal regions', False],
  ['high density', 'anal region', True],
  [None, 'pelvic fat gap', False],
  ['enlarged lymph nodes', None, False],
  ['fluid', 'Douglas pouch', True]],
 '/checkpoint/datasets.damaged/med-img-data/amosmm/imagesTrProcessed/amos_7391/text_triplet.json': [['thickening',
   'bladder',
   False],
  [None, 'uterus', False],
  ['cystic low-density lesion', None, True],
  ['enlargement', 'lymph nodes', False]],
 '/checkpoint/datasets.damage

In [26]:
enlarged_triplets = []
for filepath, triplets in chest.items():
    for trip in triplets:
        # Check each element in the triplet: if it's a string and it contains 'enlarg'
        if any(isinstance(item, str) and 'calcifications' in item.lower() for item in trip):
            enlarged_triplets.append(trip)

In [27]:
enlarged_triplets

[['calcifications', 'prostate', True],
 ['calcifications', 'prostate', False],
 ['calcifications', 'prostate', True],
 ['calcifications', 'prostate', True],
 ['calcifications', 'prostate parenchyma', True],
 ['calcifications', 'prostate', True],
 ['calcifications', 'pelvic cavity', True],
 ['calcifications', 'prostate', True],
 ['calcifications', 'prostate', True],
 ['calcifications', 'prostate', True],
 ['calcifications', 'prostate', True],
 ['calcifications', 'prostate', True],
 ['calcifications', 'prostate', True],
 ['calcifications', 'prostate', True],
 ['calcifications', 'prostate gland', True],
 ['calcifications', 'prostate', True],
 ['calcifications', 'prostate', True],
 ['calcifications', 'prostate gland', True],
 ['calcifications', 'prostate', True],
 ['calcifications', 'cyst wall', True],
 ['calcifications', 'prostate', True],
 ['calcifications', 'parenchyma', True],
 ['calcifications', 'prostate', True],
 ['calcifications', 'prostate', True],
 ['nodular calcifications', 'pro